# Data Profiling & Quality Investigation

This notebook contains the data profiling, deep profiling, and targeted investigation steps 
used to produce `docs/dataset_schema.md` and `docs/data_quality_report.md`. It verifies 
referential integrity across files, identifies missing data and its likely causes, investigates 
extreme outliers (price, availability, review counts), and diagnoses data quality issues 
(currency, neighbourhood matching, nested field formats) prior to building the cleaning pipeline.

**Scope note:** This notebook is for exploration and documentation only. The actual cleaning, 
validation, and duplicate-flagging logic that produces the pipeline's reproducible output lives 
in `src/clean.py` and `src/enrich.py`. Checks are prototyped and explained here first, then 
implemented as functions in those scripts.

This is distinct from Exploratory Data Analysis (Section 04 of the assignment), which covers 
business-facing statistical and visual analysis of the cleaned dataset — see `02_eda.ipynb`.

In [ ]:
import pandas as pd
import os

# Set paths
RAW_DATA_DIR = "../data/bronze/"
listings_path = os.path.join(RAW_DATA_DIR, "listings.csv")

# Load Data
df = pd.read_csv(listings_path, low_memory=False)
print("listings.csv loaded successfully!")

listings.csv loaded successfully!


In [4]:
print("--- 1. SHAPE & MEMORY ---")
memory_mb = df.memory_usage(deep=True).sum() / 1e6

print(f"Memory Footprint: {memory_mb:.2f} MB")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

--- 1. SHAPE & MEMORY ---
Memory Footprint: 113.33 MB
Rows: 26877
Columns: 79


In [ ]:
print("--- 2. MISSING DATA (Showing >0 missing) ---")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

# Create a DataFrame for the missing values
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0].sort_values(by='Count', ascending=False)


if not missing_df.empty:
    display(missing_df)
else:
    print("No missing data found in any column!")

--- 2. MISSING DATA (Showing >0 missing) ---


,Count,Percentage
calendar_updated,26877,100.00
neighbourhood_group_cleansed,26877,100.00
license,26748,99.52
host_neighbourhood,26696,99.33
neighborhood_overview,14045,52.26
neighbourhood,14045,52.26
host_about,13499,50.23
host_location,7070,26.31
host_response_rate,6639,24.70
host_response_time,6639,24.70


In [6]:
print("--- 3. DUPLICATE CHECKS ---")
print(f"Full row duplicates: {df.duplicated().sum()}")
if 'id' in df.columns:
    print(f"Duplicate Primary Keys ('id'): {df['id'].duplicated().sum()}")

print("\n--- 4. LOW-CARDINALITY CATEGORICALS (< 15 unique) ---")
object_cols = df.select_dtypes(include='object').columns
found_cats = False

for col in object_cols:
    n_unique = df[col].nunique()
    if 0 < n_unique < 15:   
        print(f"\n> {col} ({n_unique} unique values):")
        print(df[col].value_counts().to_string())
        found_cats = True
        
if not found_cats:
    print("No low-cardinality string columns found to display.")

--- 3. DUPLICATE CHECKS ---
Full row duplicates: 0
Duplicate Primary Keys ('id'): 0

--- 4. LOW-CARDINALITY CATEGORICALS (< 15 unique) ---

> last_scraped (2 unique values):
last_scraped
2025-09-28    15006
2025-09-29    11871

> source (2 unique values):
source
city scrape        22671
previous scrape     4206

> host_response_time (4 unique values):
host_response_time
within an hour        14118
within a few hours     3177
within a day           2264
a few days or more      679

> host_is_superhost (2 unique values):
host_is_superhost
f    17324
t     8454

> host_verifications (8 unique values):
host_verifications
['email', 'phone']                    19705
['email', 'phone', 'work_email']       4217
['phone']                              2589
['phone', 'work_email']                 248
['email']                                76
[]                                       27
['email', 'work_email']                   8
['email', 'phone', 'photographer']        1

> host_has_profile_pic

In [7]:
print("--- 5. DOMAIN SANITY CHECKS ---")
# Clean the price string (remove $ and ,) and check for negatives
if 'price' in df.columns:
    clean_price = pd.to_numeric(df['price'].astype(str).str.replace(r'[\$,]', '', regex=True), errors='coerce')
    print(f"Negative prices found: {(clean_price < 0).sum()}")
    print(f"Zero prices ($0) found: {(clean_price == 0).sum()}")

# Check for impossible availability logic
if 'minimum_nights' in df.columns and 'maximum_nights' in df.columns:
    invalid_nights = (df['minimum_nights'] > df['maximum_nights']).sum()
    print(f"Rows where minimum_nights > maximum_nights: {invalid_nights}")

--- 5. DOMAIN SANITY CHECKS ---
Negative prices found: 0
Zero prices ($0) found: 0
Rows where minimum_nights > maximum_nights: 0


In [ ]:
import pandas as pd

print("====== PROFILING: calendar.csv ======")
calendar_path = "../data/bronze/calendar.csv"
df_cal = pd.read_csv(calendar_path, low_memory=False)

print("\n--- 1. SHAPE & MEMORY ---")
print(f"Memory Footprint: {df_cal.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(f"Rows: {df_cal.shape[0]} | Columns: {df_cal.shape[1]}")

print("\n--- 2. MISSING DATA ---")
missing_cal = df_cal.isnull().sum()
missing_df_cal = pd.DataFrame({'Count': missing_cal, 'Percentage': (missing_cal / len(df_cal) * 100).round(2)})
missing_df_cal = missing_df_cal[missing_df_cal['Count'] > 0].sort_values(by='Count', ascending=False)

if not missing_df_cal.empty:
    display(missing_df_cal)
else:
    print("No missing data found!")

print("\n--- 3. DUPLICATE CHECKS ---")
print(f"Full row duplicates: {df_cal.duplicated().sum()}")
if 'listing_id' in df_cal.columns and 'date' in df_cal.columns:
    print(f"Duplicate Calendar Entries (listing_id + date): {df_cal.duplicated(subset=['listing_id', 'date']).sum()}")

print("\n--- 4. LOW-CARDINALITY CATEGORICALS (< 15 unique) ---")
object_cols_cal = df_cal.select_dtypes(include='object').columns
found_cats_cal = False

for col in object_cols_cal:
    n_unique = df_cal[col].nunique()
    if 0 < n_unique < 15:   
        print(f"\n> {col} ({n_unique} unique values):")
        print(df_cal[col].value_counts().to_string())
        found_cats_cal = True
        
if not found_cats_cal:
    print("No low-cardinality string columns found to display.")

print("\n--- 5. DOMAIN SANITY CHECKS ---")

if 'price' in df_cal.columns:
    clean_price_cal = pd.to_numeric(df_cal['price'].astype(str).str.replace(r'[\$,]', '', regex=True), errors='coerce')
    print(f"Negative prices found: {(clean_price_cal < 0).sum()}")
    print(f"Zero prices ($0) found: {(clean_price_cal == 0).sum()}")

# Check for impossible availability logic
if 'minimum_nights' in df_cal.columns and 'maximum_nights' in df_cal.columns:
    invalid_nights_cal = (df_cal['minimum_nights'] > df_cal['maximum_nights']).sum()
    print(f"Rows where minimum_nights > maximum_nights: {invalid_nights_cal}")

====== PROFILING: calendar.csv ======

--- 1. SHAPE & MEMORY ---
Memory Footprint: 1461.71 MB
Rows: 9810109 | Columns: 7

--- 2. MISSING DATA ---


,Count,Percentage
price,9810109,100.0
adjusted_price,9810109,100.0



--- 3. DUPLICATE CHECKS ---
Full row duplicates: 0
Duplicate Calendar Entries (listing_id + date): 0

--- 4. LOW-CARDINALITY CATEGORICALS (< 15 unique) ---

> available (2 unique values):
available
t    5499948
f    4310161

--- 5. DOMAIN SANITY CHECKS ---
Negative prices found: 0
Zero prices ($0) found: 0
Rows where minimum_nights > maximum_nights: 3


In [ ]:
import pandas as pd

print("====== PROFILING: reviews.csv ======")
reviews_path = "../data/bronze/reviews.csv"
df_rev = pd.read_csv(reviews_path, low_memory=False)

print("\n--- 1. SHAPE & MEMORY ---")
print(f"Memory Footprint: {df_rev.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(f"Rows: {df_rev.shape[0]} | Columns: {df_rev.shape[1]}")

print("\n--- 2. MISSING DATA ---")
missing_rev = df_rev.isnull().sum()
missing_df_rev = pd.DataFrame({'Count': missing_rev, 'Percentage': (missing_rev / len(df_rev) * 100).round(2)})
missing_df_rev = missing_df_rev[missing_df_rev['Count'] > 0].sort_values(by='Count', ascending=False)

if not missing_df_rev.empty:
    display(missing_df_rev)
else:
    print("No missing data found!")

print("\n--- 3. DUPLICATE CHECKS ---")
print(f"Full row duplicates: {df_rev.duplicated().sum()}")
if 'id' in df_rev.columns:
    print(f"Duplicate Primary Keys ('id'): {df_rev['id'].duplicated().sum()}")

print("\n--- 4. LOW-CARDINALITY CATEGORICALS (< 15 unique) ---")
object_cols_rev = df_rev.select_dtypes(include='object').columns
found_cats_rev = False

for col in object_cols_rev:
    n_unique = df_rev[col].nunique()
    if 0 < n_unique < 15:   
        print(f"\n> {col} ({n_unique} unique values):")
        print(df_rev[col].value_counts().to_string())
        found_cats_rev = True
        
if not found_cats_rev:
    print("No low-cardinality string columns found to display.")

print("\n--- 5. DOMAIN SANITY CHECKS ---")
if 'comments' in df_rev.columns:
    short_comments = df_rev['comments'].dropna().str.len().lt(5).sum()
    print(f"Short comments (< 5 characters) likely spam/useless: {short_comments}")

====== PROFILING: reviews.csv ======

--- 1. SHAPE & MEMORY ---
Memory Footprint: 327.62 MB
Rows: 664377 | Columns: 6

--- 2. MISSING DATA ---


,Count,Percentage
comments,137,0.02
reviewer_name,1,0.00



--- 3. DUPLICATE CHECKS ---
Full row duplicates: 0
Duplicate Primary Keys ('id'): 0

--- 4. LOW-CARDINALITY CATEGORICALS (< 15 unique) ---
No low-cardinality string columns found to display.

--- 5. DOMAIN SANITY CHECKS ---
Short comments (< 5 characters) likely spam/useless: 2857


## Deep Profiling
After the initial scan, we need to verify referential integrity (foreign keys) and inspect nested JSON strings.

In [ ]:
import pandas as pd
import ast
import json

RAW_DATA_DIR = "../data/bronze/"
print("====== LOADING FILES FOR DEEP PROFILING ======")

listings = pd.read_csv(f"{RAW_DATA_DIR}listings.csv", low_memory=False)
calendar = pd.read_csv(f"{RAW_DATA_DIR}calendar.csv", low_memory=False)
reviews = pd.read_csv(f"{RAW_DATA_DIR}reviews.csv", low_memory=False)
neighbourhoods = pd.read_csv(f"{RAW_DATA_DIR}neighbourhoods.csv", low_memory=False)

print(" All four datasets loaded successfully!")

====== LOADING FILES FOR DEEP PROFILING ======
 All four datasets loaded successfully!


In [13]:
print(" 1. REFERENTIAL INTEGRITY (Foreign Key Checks)")
listing_ids = set(listings["id"])
print(f"Total unique listing IDs in listings.csv: {len(listing_ids)}")

if "listing_id" in calendar.columns:
    calendar_ids = set(calendar["listing_id"])
    orphan_calendar = calendar_ids - listing_ids
    missing_calendar = listing_ids - calendar_ids
    print(f"\nCalendar listing_ids NOT found in listings.csv: {len(orphan_calendar)}")
    print(f"Listings with NO calendar entries at all:        {len(missing_calendar)}")

if "listing_id" in reviews.columns:
    review_ids = set(reviews["listing_id"])
    orphan_reviews = review_ids - listing_ids
    print(f"\nReview listing_ids NOT found in listings.csv:    {len(orphan_reviews)}")
    print(f"Distinct listings that have at least 1 review:   {len(review_ids & listing_ids)}")
    print(f"Listings with ZERO reviews:                      {len(listing_ids - review_ids)}")

 1. REFERENTIAL INTEGRITY (Foreign Key Checks)
Total unique listing IDs in listings.csv: 26877

Calendar listing_ids NOT found in listings.csv: 0
Listings with NO calendar entries at all:        0

Review listing_ids NOT found in listings.csv:    0
Distinct listings that have at least 1 review:   20753
Listings with ZERO reviews:                      6124


In [14]:
print(" 2. NEIGHBOURHOOD NAME MATCHING (listings vs neighbourhoods.csv)")
print(f"neighbourhoods.csv shape: {neighbourhoods.shape}")

nbhd_col_candidates = [c for c in neighbourhoods.columns if "neighbourhood" in c.lower()]
print(f"\nCandidate neighbourhood columns in neighbourhoods.csv: {nbhd_col_candidates}")

if "neighbourhood_cleansed" in listings.columns and nbhd_col_candidates:
    nbhd_col = nbhd_col_candidates[0]
    listing_nbhds = set(listings["neighbourhood_cleansed"].dropna())
    ref_nbhds = set(neighbourhoods[nbhd_col].dropna())

    unmatched = listing_nbhds - ref_nbhds
    print(f"\nUnique neighbourhoods in listings.csv:        {len(listing_nbhds)}")
    print(f"Unique neighbourhoods in neighbourhoods.csv:  {len(ref_nbhds)}")
    print(f"Neighbourhoods in listings but NOT in ref file: {len(unmatched)}")
    if unmatched:
        print(f"  Sample unmatched names: {list(unmatched)[:10]}")

 2. NEIGHBOURHOOD NAME MATCHING (listings vs neighbourhoods.csv)
neighbourhoods.csv shape: (116, 2)

Candidate neighbourhood columns in neighbourhoods.csv: ['neighbourhood_group', 'neighbourhood']

Unique neighbourhoods in listings.csv:        87
Unique neighbourhoods in neighbourhoods.csv:  0
Neighbourhoods in listings but NOT in ref file: 87
  Sample unmatched names: ['Ward 4', 'Ward 8', 'Ward 97', 'Ward 45', 'Ward 61', 'Ward 44', 'Ward 66', 'Ward 5', 'Ward 113', 'Ward 72']


In [15]:
print(" 3. NESTED FIELD INSPECTION (amenities, host_verifications)")

for col in ["amenities", "host_verifications"]:
    if col in listings.columns:
        sample = listings[col].dropna().iloc[0]
        print(f"\n> {col} raw sample:\n{sample[:200]}... [truncated]")
        
        parsed = None
        try:
            parsed = json.loads(sample)
            print("   Parses successfully with json.loads()")
        except Exception:
            try:
                parsed = ast.literal_eval(sample)
                print("   Parses successfully with ast.literal_eval() (Python list repr)")
            except Exception as e:
                print(f"   Could not parse with either method: {e}")

        if parsed is not None:
            print(f"  Parsed type: {type(parsed)} | Length: {len(parsed)}")

        # Amenity Counter
        if col == "amenities":
            try:
                all_amenities = set()
                for val in listings[col].dropna().head(2000): 
                    try:
                        items = json.loads(val)
                    except Exception:
                        items = ast.literal_eval(val)
                    all_amenities.update(items)
                print(f"  Distinct amenities found in a 2000-row sample: {len(all_amenities)}")
            except Exception as e:
                print(f"  Could not aggregate amenities sample: {e}")

 3. NESTED FIELD INSPECTION (amenities, host_verifications)

> amenities raw sample:
["Bathtub", "Fire extinguisher", "Oven", "Portable fans", "Portable heater", "Beach essentials", "Smoke alarm", "Carbon monoxide alarm", "First aid kit", "Lockbox", "Drying rack for clothing", "Dedica... [truncated]
   Parses successfully with json.loads()
  Parsed type: <class 'list'> | Length: 68
  Distinct amenities found in a 2000-row sample: 1563

> host_verifications raw sample:
['email', 'phone', 'work_email']... [truncated]
   Parses successfully with ast.literal_eval() (Python list repr)
  Parsed type: <class 'list'> | Length: 3


In [16]:
print(" 4. bathrooms (numeric) vs bathrooms_text (string) COMPARISON")

if "bathrooms" in listings.columns and "bathrooms_text" in listings.columns:
    print(f"'bathrooms' missing:      {listings['bathrooms'].isnull().sum()} ({listings['bathrooms'].isnull().mean()*100:.2f}%)")
    print(f"'bathrooms_text' missing: {listings['bathrooms_text'].isnull().sum()} ({listings['bathrooms_text'].isnull().mean()*100:.2f}%)")

    mask = listings["bathrooms"].isnull() & listings["bathrooms_text"].notnull()
    
    print("\nSample rows where 'bathrooms' is null but 'bathrooms_text' is NOT null:")
    display(listings.loc[mask, ["bathrooms", "bathrooms_text"]].head(10))

    print(f"\n>> Rows where bathrooms is null but bathrooms_text has data: {mask.sum()}")
    print("   If this number is high, bathrooms_text is the more reliable source.")

 4. bathrooms (numeric) vs bathrooms_text (string) COMPARISON
'bathrooms' missing:      4372 (16.27%)
'bathrooms_text' missing: 236 (0.88%)

Sample rows where 'bathrooms' is null but 'bathrooms_text' is NOT null:


,bathrooms,bathrooms_text
4,NaN,1 bath
8,NaN,2 baths
10,NaN,1.5 baths
11,NaN,1 bath
12,NaN,1 bath
13,NaN,1 bath
15,NaN,2.5 baths
16,NaN,2 baths
33,NaN,1 bath
45,NaN,2 baths



>> Rows where bathrooms is null but bathrooms_text has data: 4150
   If this number is high, bathrooms_text is the more reliable source.


In [17]:
print(" 5. DATE RANGE SANITY CHECKS")

if "host_since" in listings.columns:
    host_since = pd.to_datetime(listings["host_since"], errors="coerce")
    print(f"host_since range: {host_since.min()}  →  {host_since.max()}")
    print(f"Unparseable host_since dates: {host_since.isnull().sum() - listings['host_since'].isnull().sum()}")

if "date" in calendar.columns:
    cal_dates = pd.to_datetime(calendar["date"], errors="coerce")
    print(f"\ncalendar.csv date range: {cal_dates.min()}  →  {cal_dates.max()}")
    span_days = (cal_dates.max() - cal_dates.min()).days
    print(f"Span: {span_days} days (expect ~365 for a forward-looking calendar)")

if "date" in reviews.columns:
    rev_dates = pd.to_datetime(reviews["date"], errors="coerce")
    print(f"\nreviews.csv date range: {rev_dates.min()}  →  {rev_dates.max()}")

 5. DATE RANGE SANITY CHECKS
host_since range: 2009-07-11 00:00:00  →  2025-09-26 00:00:00
Unparseable host_since dates: 0

calendar.csv date range: 2025-09-28 00:00:00  →  2026-09-28 00:00:00
Span: 365 days (expect ~365 for a forward-looking calendar)

reviews.csv date range: 2010-06-15 00:00:00  →  2025-09-28 00:00:00


In [18]:
print(" 6. OUTLIER INSPECTION")

if "price" in listings.columns:
    price_clean = pd.to_numeric(
        listings["price"].astype(str).str.replace(r"[\$,]", "", regex=True), errors="coerce"
    )
    print("Price distribution (cleaned):")
    print(price_clean.describe())

    print("\nTop 10 highest-priced listings:")
    top_idx = price_clean.nlargest(10).index
    cols_to_show = [c for c in ["id", "room_type", "accommodates", "neighbourhood_cleansed"] if c in listings.columns]
    display_df = listings.loc[top_idx, cols_to_show].copy()
    display_df["price_clean"] = price_clean.loc[top_idx]
    display(display_df)

if "minimum_nights" in listings.columns:
    extreme_min_nights = listings[listings["minimum_nights"] > 365]
    print(f"\nListings with minimum_nights > 365: {len(extreme_min_nights)}")

if "minimum_nights" in calendar.columns and "maximum_nights" in calendar.columns:
    bad_rows = calendar[calendar["minimum_nights"] > calendar["maximum_nights"]]
    print(f"\ncalendar.csv rows where minimum_nights > maximum_nights: {len(bad_rows)}")
    if len(bad_rows) > 0:
        display(bad_rows.head())

 6. OUTLIER INSPECTION
Price distribution (cleaned):
count     22476.000000
mean       3280.999778
std        9048.338102
min         161.000000
25%         950.000000
50%        1526.000000
75%        2975.250000
max      714885.000000
Name: price, dtype: float64

Top 10 highest-priced listings:


,id,room_type,accommodates,neighbourhood_cleansed,price_clean
25236,1450625142959535977,Entire home/apt,4,Ward 59,714885.0
2260,13746859,Entire home/apt,4,Ward 62,323000.0
20332,1267956384324715876,Entire home/apt,6,Ward 54,297812.0
20333,1267956410077042168,Entire home/apt,6,Ward 54,255267.0
22913,1346064572109464832,Entire home/apt,1,Ward 35,228467.0
21646,1299776803631524660,Entire home/apt,14,Ward 54,184000.0
22914,1346069309879863168,Entire home/apt,10,Ward 116,173866.0
23819,1387405539167433382,Entire home/apt,8,Ward 62,162471.0
23820,1387419954213600966,Entire home/apt,3,Ward 69,162471.0
23821,1387426761848426117,Entire home/apt,2,Ward 15,162471.0



Listings with minimum_nights > 365: 8

calendar.csv rows where minimum_nights > maximum_nights: 3


,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
1482089,22780052,2026-05-31,t,NaN,NaN,2,1
1681545,22173943,2026-05-31,t,NaN,NaN,2,1
8135639,1329692465987536105,2026-01-29,f,NaN,NaN,2,1


In [19]:
print(" 7. reviews.csv ADDITIONAL CHECKS")

if reviews is not None:
    print(f"Distinct listings with at least one review: {reviews['listing_id'].nunique()}")
    
    if 'reviewer_id' in reviews.columns:
        print(f"Distinct reviewers: {reviews['reviewer_id'].nunique()}")
        
    if "comments" in reviews.columns:
        empty_comments = reviews["comments"].isnull().sum()
        very_short = reviews["comments"].dropna().str.len().lt(5).sum()
        print(f"Null comments: {empty_comments}")
        print(f"Comments shorter than 5 characters (likely low-quality/spam): {very_short}")

print(f"\n{'='*70}")
print(" DEEP PROFILING COMPLETE")
print(f"{'='*70}")

 7. reviews.csv ADDITIONAL CHECKS
Distinct listings with at least one review: 20753
Distinct reviewers: 452588
Null comments: 137
Comments shorter than 5 characters (likely low-quality/spam): 2857

 DEEP PROFILING COMPLETE


## Targeted Investigation
The deep profiling revealed extreme outliers (prices > 700k) and potential neighbourhood matching issues. We investigate the raw strings to find the root cause.

In [ ]:
import pandas as pd

print("====== TARGETED INVESTIGATION ======")

listings = pd.read_csv("../data/bronze/listings.csv", low_memory=False)
nbhd = pd.read_csv("../data/bronze/neighbourhoods.csv")

print("\n--- 1. RAW PRICE STRING CHECK ---")
print("First 10 raw price values from listings.csv:")

print(listings['price'].dropna().head(10).tolist())

====== TARGETED INVESTIGATION ======

--- 1. RAW PRICE STRING CHECK ---
First 10 raw price values from listings.csv:
['$2,315.00', '$4,785.00', '$1,118.00', '$2,250.00', '$621.00', '$51,429.00', '$1,641.00', '$700.00', '$7,663.00', '$1,649.00']


In [ ]:
print("--- 2. EXTREME PRICE OUTLIERS vs MINIMUM NIGHTS ---")


listings['price_clean'] = pd.to_numeric(
    listings['price'].astype(str).str.replace(r"[\$,]", "", regex=True), 
    errors="coerce"
)

print("\nTop 10 most expensive listings:")
top_10 = listings.nlargest(10, 'price_clean')[['id', 'price_clean', 'minimum_nights', 'room_type']]
display(top_10)

print("\nListings requiring more than 365 nights minimum:")
extreme_nights = listings[listings['minimum_nights'] > 365][['id', 'price_clean', 'minimum_nights']]
display(extreme_nights)

--- 2. EXTREME PRICE OUTLIERS vs MINIMUM NIGHTS ---

Top 10 most expensive listings:


,id,price_clean,minimum_nights,room_type
25236,1450625142959535977,714885.0,1,Entire home/apt
2260,13746859,323000.0,3,Entire home/apt
20332,1267956384324715876,297812.0,3,Entire home/apt
20333,1267956410077042168,255267.0,3,Entire home/apt
22913,1346064572109464832,228467.0,1,Entire home/apt
21646,1299776803631524660,184000.0,3,Entire home/apt
22914,1346069309879863168,173866.0,1,Entire home/apt
23819,1387405539167433382,162471.0,1,Entire home/apt
23820,1387419954213600966,162471.0,1,Entire home/apt
23821,1387426761848426117,162471.0,1,Entire home/apt



Listings requiring more than 365 nights minimum:


,id,price_clean,minimum_nights
876,6084051,NaN,700
1587,9781581,NaN,699
2330,14146609,NaN,730
6108,33109278,NaN,999
8064,45278530,502.0,730
19733,1242818287422338372,800.0,999
19734,1242819235339113998,1000.0,999
21431,1294436095188244113,483.0,999


In [22]:
print("--- 3. NEIGHBOURHOOD MATCHING BUG DIAGNOSIS ---")

print("A. Data Types in neighbourhoods.csv:")
print(nbhd.dtypes)

print("\nB. Checking for hidden nulls or weird spacing in neighbourhoods.csv:")
#
print(repr(nbhd['neighbourhood'].head(5).tolist()))

print("\nC. Checking listings.csv format for comparison:")
print(repr(listings['neighbourhood_cleansed'].dropna().head(5).tolist()))

print("\n INVESTIGATION COMPLETE. The notebook is now fully documented.")

--- 3. NEIGHBOURHOOD MATCHING BUG DIAGNOSIS ---
A. Data Types in neighbourhoods.csv:
neighbourhood_group    float64
neighbourhood           object
dtype: object

B. Checking for hidden nulls or weird spacing in neighbourhoods.csv:
['Ward 1', 'Ward 10', 'Ward 100', 'Ward 101', 'Ward 102']

C. Checking listings.csv format for comparison:
['Ward 23', 'Ward 23', 'Ward 4', 'Ward 115', 'Ward 112']

 INVESTIGATION COMPLETE. The notebook is now fully documented.


## Availability & Review Count Outliers (the new gap-fill checks)

In [1]:
import pandas as pd

print("====== 8. AVAILABILITY & REVIEW OUTLIERS (Section 3.1) ======")
listings = pd.read_csv("../data/bronze/listings.csv", low_memory=False)

print("\n--- AVAILABILITY 365 ---")
print(listings['availability_365'].describe())
print(f"Listings with availability = 0 (Likely inactive): {(listings['availability_365'] == 0).sum()}")
print(f"Listings with availability = 365 (Fully open):    {(listings['availability_365'] == 365).sum()}")

print("\n--- REVIEW COUNT OUTLIERS ---")
print(listings['number_of_reviews'].describe())
print("\nTop 10 Most Reviewed Listings:")
top_reviewed = listings.nlargest(10, 'number_of_reviews')[
    ['id', 'number_of_reviews', 'room_type', 'host_is_superhost']
]
display(top_reviewed)

====== 8. AVAILABILITY & REVIEW OUTLIERS (Section 3.1) ======

--- AVAILABILITY 365 ---
count    26877.000000
mean       204.592068
std        124.557137
min          0.000000
25%         90.000000
50%        236.000000
75%        316.000000
max        365.000000
Name: availability_365, dtype: float64
Listings with availability = 0 (Likely inactive): 3209
Listings with availability = 365 (Fully open):    1032

--- REVIEW COUNT OUTLIERS ---
count    26877.000000
mean        24.719165
std         48.541829
min          0.000000
25%          1.000000
50%          6.000000
75%         26.000000
max        843.000000
Name: number_of_reviews, dtype: float64

Top 10 Most Reviewed Listings:


,id,number_of_reviews,room_type,host_is_superhost
601,4646142,843,Entire home/apt,NaN
1934,11540101,785,Entire home/apt,t
975,6711928,706,Entire home/apt,f
2720,15733201,705,Entire home/apt,NaN
5244,28656609,703,Entire home/apt,t
3357,17851693,587,Private room,t
6774,38475168,562,Entire home/apt,t
3467,18517007,543,Entire home/apt,t
319,2655966,542,Entire home/apt,t
1329,8860800,534,Entire home/apt,t


## Fuzzy Duplicate Detection 

In [ ]:
import pandas as pd
import numpy as np

print("====== 9. FUZZY DUPLICATE DETECTION (Section 3.1) ======")
listings = pd.read_csv("../data/bronze/listings.csv", low_memory=False)


exact_dupes = listings.duplicated().sum()
id_dupes = listings['id'].duplicated().sum()
print(f"Deterministic (Exact Row) Duplicates: {exact_dupes}")
print(f"Deterministic (Primary Key) Duplicates: {id_dupes}")


def haversine_km(lat1, lon1, lat2, lon2):
    """Calculates the great-circle distance between two points on a sphere."""
    R = 6371 # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

print("\nRunning geospatial fuzzy matching (Same host, <50m apart)...")
distance_threshold_km = 0.05 # 50 meters
potential_dupes = []


for host_id, group in listings.groupby('host_id'):
    if len(group) < 2:
        continue # Skip hosts with only 1 listing
        
    coords = group[['latitude', 'longitude']].values
    ids = group['id'].values
    
    # Compare every listing against every other listing for this host
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            dist = haversine_km(*coords[i], *coords[j])
            if dist < distance_threshold_km:
                potential_dupes.append({
                    'host_id': host_id,
                    'listing_1': ids[i],
                    'listing_2': ids[j],
                    'distance_km': round(dist, 4)
                })

print(f"Fuzzy duplicate candidates found: {len(potential_dupes)}")

if potential_dupes:
    dupe_df = pd.DataFrame(potential_dupes)
    print("\nSample of fuzzy duplicate pairs:")
    display(dupe_df.head(10))
    print("\nConclusion: These are likely legitimate multi-unit properties (e.g., separate rooms in one guesthouse).")
    print("Action: They will be flagged in the Silver layer, but NOT deleted.")

====== 9. FUZZY DUPLICATE DETECTION (Section 3.1) ======
Deterministic (Exact Row) Duplicates: 0
Deterministic (Primary Key) Duplicates: 0

Running geospatial fuzzy matching (Same host, <50m apart)...
Fuzzy duplicate candidates found: 11956

Sample of fuzzy duplicate pairs:


,host_id,listing_1,listing_2,distance_km
0,59342,15077,1643095,0.0000
1,59342,15077,1689925,0.0000
2,59342,15077,1706299,0.0000
3,59342,15077,13231393,0.0000
4,59342,15077,591023924079194675,0.0389
5,59342,1643095,1689925,0.0000
6,59342,1643095,1706299,0.0000
7,59342,1643095,13231393,0.0000
8,59342,1643095,591023924079194675,0.0389
9,59342,1689925,1706299,0.0000



Conclusion: These are likely legitimate multi-unit properties (e.g., separate rooms in one guesthouse).
Action: They will be flagged in the Silver layer, but NOT deleted.


In [3]:
import pandas as pd
import numpy as np

# Load the data
listings = pd.read_csv("../data/bronze/listings.csv", low_memory=False)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

potential_dupes = []
for host_id, group in listings.groupby('host_id'):
    if len(group) < 2: continue
    coords = group[['latitude', 'longitude']].values
    ids = group['id'].values
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            dist = haversine_km(*coords[i], *coords[j])
            if dist < 0.05:
                potential_dupes.append({'host_id': host_id, 'listing_1': ids[i], 'listing_2': ids[j]})

fuzzy_df = pd.DataFrame(potential_dupes)

print("--- DIAGNOSING THE 11,956 PAIRS ---")
print(f"Total Unique Hosts responsible for all pairs: {fuzzy_df['host_id'].nunique()}")
print(f"Total Unique Listings involved in the pairs: {pd.concat([fuzzy_df['listing_1'], fuzzy_df['listing_2']]).nunique()}")

# Let's find the worst offenders
print("\nTop 5 Hosts by number of candidate pairs:")
print(fuzzy_df['host_id'].value_counts().head(5))

--- DIAGNOSING THE 11,956 PAIRS ---
Total Unique Hosts responsible for all pairs: 1272
Total Unique Listings involved in the pairs: 5696

Top 5 Hosts by number of candidate pairs:
host_id
677109474    1467
656775352     990
488733978     904
551075151     819
659985038     694
Name: count, dtype: int64
